<a href="https://colab.research.google.com/github/AKSHAYA-1412/GEN-AI-TRAINING/blob/main/Assignment_GEN_AI_1_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch torchvision matplotlib scikit-learn

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

#transform Converts images into tensors and normalize pixel values to make them suitable for training a neural network
# a tensor is a container of numbers ,basically numerical representaion of images
#normalise - scaling values to a specific range to improve model training
transform = transforms.Compose([
    transforms.ToTensor(), # if image pixel is [0 50 255] after tensor it is [0.0 0.2 0.1] as it is divided by 255
    transforms.Normalize((0.5,), (0.5,)) # after normalisation it is -1 to +1
])
#trains dataset ,helps model to learn features like curves
train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)


100%|██████████| 9.91M/9.91M [00:00<00:00, 35.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.70MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.9MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.5MB/s]


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)#kernal small box filter that scans images to find patterns , 32 refers to filters , 1 input layer
        #basic edges
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)#32 filters in first layer is converted to 32 inputs in second layer
        #complex shapes

        self.pool = nn.MaxPool2d(2, 2)#reduces the size of the image

        self.fc1 = nn.Linear(64 * 5 * 5, 128)#64 feature images ,each of 5*5 size,
        self.fc2 = nn.Linear(128, 10)#classsification

        self.dropout = nn.Dropout(0.25)#turns off some neurons to prevent overfittling-model memorises the training data instad of learning from it

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))#Relu removes negative value (conv--. relu--.pool--.flatten then fc)
        x = self.pool(F.relu(self.conv2(x)))

        x = x.view(-1, 64 * 5 * 5)

        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)

        return x

model = CNN()

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss() # predicts how wrong the model is
optimizer = optim.Adam(model.parameters(), lr=0.001)# if model makes mistake it fixes it

epochs = 5 # model will learn 5 times

for epoch in range(epochs):
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:# will load 64 images

        optimizer.zero_grad()#clear old gradients - gradient tells how to change weights to reduce error
        outputs = model(images)#model predicts
        loss = criterion(outputs, labels)#compare prediction vs actual

        loss.backward()#model learns from mistake
        optimizer.step()#fix weights--are numbers how important each feature is

        running_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Epoch {epoch+1}, Loss: {running_loss:.3f}, Accuracy: {100 * correct / total:.2f}%")

Epoch 1, Loss: 163.611, Accuracy: 94.69%
Epoch 2, Loss: 54.158, Accuracy: 98.29%
Epoch 3, Loss: 38.489, Accuracy: 98.74%
Epoch 4, Loss: 28.693, Accuracy: 99.04%
Epoch 5, Loss: 24.537, Accuracy: 99.16%


In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

#switch model to test mode
model.eval()
all_preds = []#store predictions
all_labels = []#store actual values

correct = 0
total = 0

with torch.no_grad():#turn of learning
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        all_preds.extend(predicted.numpy())
        all_labels.extend(labels.numpy())

print(f"Test Accuracy: {100 * correct / total:.2f}%")#it is calculated by comparing predictions with actual value

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:\n", cm)#table that is used to evaluate how well a classification model is performing by showing correct and incorrect predictions

Test Accuracy: 98.97%
Confusion Matrix:
 [[ 976    0    1    0    0    0    0    1    2    0]
 [   0 1126    1    2    0    0    4    1    1    0]
 [   1    0 1027    0    0    0    1    2    1    0]
 [   0    0    5 1001    0    3    0    0    1    0]
 [   0    0    0    0  965    0    4    0    2   11]
 [   2    0    0    4    0  882    1    1    0    2]
 [   5    1    0    0    1    1  950    0    0    0]
 [   0    2    5    0    0    0    0 1017    0    4]
 [   2    0    4    0    0    0    0    0  967    1]
 [   1    0    1    0    2    3    0    3   13  986]]


In [ ]:
# Filters
conv1 = nn.Conv2d(1, 64, 3)# changing values to improve models

# Kernel size
nn.Conv2d(1, 32, kernel_size=5)

# Dropout
dropout = nn.Dropout(0.5)

# Learning rate
optimizer = optim.Adam(model.parameters(), lr=0.0005)
